<p><font size="6" color='grey'> <b>

Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
Multimodales RAG
</b></font> </br></p>


---

<p><font color='darkblue' size="4">
ℹ️ <b>Hinweis</b>
</font></p>

> Dieses Notebook nutzt das **`genai_lib.multimodal_rag`-Modul**, das speziell für multimodale RAG-Pipelines entwickelt wurde.


In [1]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul

# ── Projekt-Utilities ─────────────────────────────────────────────────────────
from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt
)
setup_api_keys(['OPENAI_API_KEY', 'HF_TOKEN'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ HF_TOKEN erfolgreich gesetzt

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.3.9
langchain-chroma                         1.1.0
langchain-classic                        1.0.8
langchain-community                      0.4.2
langchain-core                           1.4.7
langchain-ollama                         1.1.0
langchain-openai                         1.3.3
langchain-protocol                       0.0.17
langchain-text-splitters                 1.1.2
langgraph                                1.2.5
langgraph-checkpoint                     4.1.1
langgraph-checkpoint-sqlite              3.1.0
langgraph-prebuilt                       1.1.0
langgraph-sdk                            0.4.2

IP-Adresse: 34.150.177.150
Hostname: 150.177.150.34.bc.googleusercontent.com
Stadt: Washington
Region: District of Columbia
Land: US
Koordinaten: 38.89

In [2]:
#@title 🛠️ Installationen { display-mode: "form" }
install_packages([
    ('markitdown[all]', 'markitdown'),
])

✅ markitdown bereits verfügbar


In [3]:
#@title 📂 Dokumente, Bilder { display-mode: "form" }

# ── Stdlib ────────────────────────────────────────────────────────────────────
import shutil

# ── Projekt-Utilities ─────────────────────────────────────────────────────────
from genai_lib.utilities import copy_from_github

shutil.rmtree("files", ignore_errors=True)

# --- Texte
copy_from_github(source="ralf-42/GenAI/02_daten/01_text", target="files", mask="biografien_*",)
copy_from_github(source="ralf-42/GenAI/02_daten/01_text", target="files", mask="roboter.txt",)

# --- Bilder (nach files/)
copy_from_github(source="ralf-42/GenAI/02_daten/02_bild", target="files", mask="retro_robot.jpg",)
copy_from_github(source="ralf-42/GenAI/02_daten/02_bild", target="files", mask="hedra_cyborg.png",)
copy_from_github(source="ralf-42/GenAI/02_daten/02_bild", target="files", mask="apfel.png",)

# --- Bild (Arbeitsverzeichnis)
copy_from_github(source="ralf-42/GenAI/02_daten/02_bild", target=".", mask="zwei_roboter.png",)

  Kopiere: 02_daten/01_text/biografien_1.txt → files/biografien_1.txt
  Kopiere: 02_daten/01_text/biografien_2.md → files/biografien_2.md
  Kopiere: 02_daten/01_text/biografien_3.pdf → files/biografien_3.pdf
  Kopiere: 02_daten/01_text/biografien_4.docx → files/biografien_4.docx

Ergebnis: 4 Datei(en) kopiert → files
  Kopiere: 02_daten/01_text/roboter.txt → files/roboter.txt

Ergebnis: 1 Datei(en) kopiert → files
  Kopiere: 02_daten/02_bild/retro_robot.jpg → files/retro_robot.jpg

Ergebnis: 1 Datei(en) kopiert → files
  Kopiere: 02_daten/02_bild/hedra_cyborg.png → files/hedra_cyborg.png

Ergebnis: 1 Datei(en) kopiert → files
  Kopiere: 02_daten/02_bild/apfel.png → files/apfel.png

Ergebnis: 1 Datei(en) kopiert → files
  Kopiere: 02_daten/02_bild/zwei_roboter.png → zwei_roboter.png

Ergebnis: 1 Datei(en) kopiert → .


['zwei_roboter.png']

# 1 | RAG-Prozess
---

In dem entwickelten **multimodalen Retrieval-Augmented-Generation (RAG)**-System werden sowohl Text- als auch Bilddaten verarbeitet.
Für die **Bildverarbeitung** kommt ein zweistufiger Ansatz zum Einsatz:

1. **Image-Embedding:**
   Das Bild wird in einen hochdimensionalen Vektorraum eingebettet, um visuelle Merkmale wie Formen, Farben und Strukturen numerisch zu repräsentieren.

2. **Textbeschreibung und Text-Embedding:**
   Zusätzlich wird mit *gpt-4o-mini* eine sprachliche Beschreibung des Bildes erzeugt. Diese Beschreibung wird anschließend in ein Text-Embedding überführt, um semantische Informationen textbasiert nutzbar zu machen.

---

**Vorteile des Ansatzes:**

* **Erweiterte semantische Repräsentation:**
  Durch die Kombination von visuellen und sprachlichen Embeddings werden sowohl konkrete als auch konzeptuelle Eigenschaften eines Bildes abgebildet.

* **Verbesserte Retrieval-Qualität:**
  Textbasierte Suchanfragen können nicht nur über visuelle Ähnlichkeiten, sondern auch über die semantisch beschriebene Bedeutung der Bilder beantwortet werden.

* **Höhere Interpretierbarkeit:**
  Die generierte Bildbeschreibung ermöglicht eine transparente Nachvollziehbarkeit der zugrunde liegenden Repräsentationen und unterstützt bei der Evaluierung der Ergebnisse.

* **Gute Erweiterbarkeit:**
  Das Verfahren ist modular aufgebaut und lässt sich leicht um weitere Modalitäten wie Audio oder Video ergänzen.


<p><font color='black' size="5">
RAG-Prozess für Texte
</font></p>

<img src="https://raw.githubusercontent.com/ralf-42/GenAI/main/07_image/rag_process.png" width="600" alt="Avatar">



<p><font color='black' size="5">
RAG-Prozess für Bilder
</font></p>

<img src="https://raw.githubusercontent.com/ralf-42/GenAI/main/07_image/rag_process_03.png" width="600" alt="Avatar">



# 2 | Modul `multimodal_rag`
---

Python-Modul für ein **funktionales multimodales RAG-System**, das Text- und Bilddokumente in einer einheitlichen Vektordatenbank verwaltet und durchsucht.

**Modalitätsrichtungen**

| Eingabe (Query) | Ausgabe (Antwort) | Beispiel / Beschreibung | Status |
|-----------------|-------------------|-------------------------|--------|
| **Text → Text** | Textbasierte Frage führt zu Textantwort | Klassisches RAG-System (z.B. Chatbot, Q&A) | ✅ |
| **Text → Bild** | Textanfrage findet relevante Bilder | "Zeige mir Roboter-Bilder" | ✅ |
| **Bild → Text** | Bildanalyse oder Captioning | "Was ist auf diesem Foto zu sehen?" | ✅ |
| **Bild → Bild** | Bildretrieval oder visuelle Transformation | "Finde ähnliche Bilder" | ✅ |
| **Bild → Text/Bild** | Erweiterte multimodale Suche mit Bild | "Alle Infos zu diesem Bild" (CLIP + Semantik + Cross-Modal) | ✅ |
| **Text + Bild → Text** | Kombination zur Textgenerierung | "Welche Informationen enthält dieses Diagramm?" | ❌ |
| **Text + Bild → Bild** | Bedingte Bildgenerierung | "Mach aus diesem Bild eine Winterversion" | ❌ |

**Hauptvorteile**

1. **Funktionale Architektur**: Klare Trennung von Konfiguration, Komponenten und Funktionen
2. **Einheitliche Datenbank**: ChromaDB mit separaten Collections für Text und Bilder
3. **Hybride Suche**: Text-Embeddings (OpenAI) + Bild-Embeddings (CLIP)
4. **Flexible Konfiguration**: Alle Parameter über RAGConfig anpassbar
5. **Bildbeschreibung**: Zu jedem Bild wird zusätzlich eine Bildbeschreibung erstellt.

<p><font color='black' size="5">
Aufbau
</font></p>

**RAGConfig**
- Zentrale Konfigurationsklasse
- Anpassbare Parameter (chunk_size, models, thresholds)

**RAGComponents**
- Container für alle System-Komponenten
- Text-Embeddings, CLIP-Model, LLMs, Collections

**Hauptfunktionen**
- Datensammlung
    - `init_rag_system()`: System-Initialisierung
    - `process_directory()`: Bulk-Import von Dateien
    - `add_text_document()`: Einzelnes Dokument hinzufügen
    - `add_image_with_description()`: Bild mit Auto-Beschreibung
- Abruf & Erweiterung
    - `search_texts()`: Text-Suche inkl. Bildbeschreibungen
    - `search_images()`: CLIP-basierte Bildsuche
    - `search_similar_images()`: Bild→Bild Ähnlichkeitssuche
    - `search_text_by_image()`: Bild→Text Suche
    - `multimodal_search()`: Erweiterte multimodale Suche (Text-Query)
    - `multimodal_search_by_image()`: Erweiterte multimodale Suche (Bild-Query)

In [6]:
#@markdown   <p><font size="4" color='green'> Multimodal-Prozess</font> </br></p>


diagram = """
%%{init: {'theme':'light'}}%%
flowchart TB
    User(("Benutzer"))

    subgraph Init["<b>Systeminitialisierung</b>"]
        direction TB
        RAGConfig["RAGConfig<br/>Konfiguration"]
        init_rag["init_rag_system"]
        RAGComponents["RAGComponents<br/>Text-Embeddings, CLIP,<br/>LLMs, Collections"]

        RAGConfig --> init_rag
        init_rag --> RAGComponents
    end

    subgraph Generate["<b>Datensammlung</b>"]
        direction TB
        process_dir["process_directory"]
        add_text["add_text_document"]
        add_image["add_image_with_description"]
        generate_desc["generate_image_description<br/>Vision LLM"]

        process_dir --> add_text
        process_dir --> add_image
        add_image --> generate_desc
    end

    subgraph Search["<b>Suchfunktionen</b>"]
        direction TB
        multimodal["multimodal_search<br/>Kombinierte Suche"]
        search_text_by_img["search_text_by_image<br/>Bild → Text"]
        search_images["search_images<br/>Text → Bild"]
        search_texts["search_texts<br/>Text → Text"]
        search_similar["search_similar_images<br/>Bild → Bild"]

        multimodal --> search_images
        multimodal --> search_texts
        multimodal --> search_similar
        multimodal --> search_text_by_img
    end

    subgraph DB["<b>Datenbanken & Modelle</b>"]
        direction LR
        text_coll[("Text Collection<br/>ChromaDB")]
        image_coll[("Image Collection<br/>ChromaDB")]
        clip["CLIP Model<br/>ViT-B-32"]
        llm["LLM<br/>gpt-4o-mini"]

        text_coll ~~~ image_coll ~~~ clip ~~~ llm
    end

    %% Benutzer-Verbindungen
    User -->|"#1 Init"| Init
    User -->|"#2 Generate"| Generate
    User -->|"#3 Search"| Search

    %% Init zu DB
    RAGComponents -.->|"initialisiert"| DB

    %% Generate zu DB
    Generate -.->|"speichert"| DB

    %% Search zu DB
    Search -.->|"abfragt"| DB

    %% Styling
    style Init fill:#ffebee,stroke:#c62828,stroke-width:2px
    style Generate fill:#fff3e0,stroke:#e65100,stroke-width:2px
    style Search fill:#f3e5f5,stroke:#7b1fa2,stroke-width:2px
    style DB fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px
    style User fill:#cddc39,stroke:#827717,stroke-width:2px
"""
mermaid(diagram, 1100)


In [7]:
# Imports

# ── Projekt-Utilities ─────────────────────────────────────────────────────────
from genai_lib.multimodal_rag import (
    # Init
    init_rag_system,
    get_system_status,

    # Generate
    process_directory,
    add_text_document,
    add_image_with_description,

    # Search
    multimodal_search,
    multimodal_search_by_image,
    search_texts,
    search_images,
    search_text_by_image,
    search_similar_images,
)

DATA_DIR = "./files"

ImportError: cannot import name '_Ink' from 'PIL._typing' (/usr/local/lib/python3.12/dist-packages/PIL/_typing.py)

# 3 | RAG initialisieren
---

In [ ]:
# Initialisierung
rag = init_rag_system()

In [ ]:
# Konfiguration abfragen
print(rag.config)

# Konfiguration ändern (Beispiel: chunk_size ändern)
# rag.config.chunk_size = 300

# 4 | Datensammlung
---

In [ ]:
# Daten laden und verarbeiten
# Hinweis: auto_describe_images=True kann Vision-/API-Kosten verursachen.
process_directory(rag, DATA_DIR, auto_describe_images=True)

# 5 | Suche mit Text
---

<p><font color='black' size="5">
Text → Text
</font></p>

In [ ]:
result = search_texts(rag, "Was weisst Du über Cyborgs?")
mprint(result)

<p><font color='black' size="5">
Text → Bild
</font></p>

In [ ]:
result = search_images(rag, "Was weisst Du über Cyborgs?")
mprint(result)

<p><font color='black' size="5">
Text → Text/Bild
</font></p>

In [ ]:
result = multimodal_search(rag, "Was weisst Du über Cyborgs?")
mprint(result)

# 6 | Suche mit Bild
---

<p><font color='black' size="5">
Bild → Bild
</font></p>

In [ ]:
result = search_similar_images(rag, "./zwei_roboter.png", k=5)
mprint("## 🖼️ Suche Bild → Bild")
mprint("---")
for img in result:
    mprint(f"{img['filename']}: Ähnlichkeit: {img['similarity']}")

<p><font color='black' size="5">
Bild → Text
</font></p>

In [ ]:
result = search_text_by_image(rag, "./zwei_roboter.png", k=5)
mprint(result)

<p><font color='black' size="5">
Bild → Text/Bild
</font></p>

In [ ]:
result = multimodal_search_by_image(rag, "./zwei_roboter.png", k_similar_images=3, k_text=3, k_related_images=2)
mprint(result)

# A | Aufgabe
---



**Grundlagen**

Lade ein multimodales Dokument und beantworte eine einfache Frage darüber:

1. Lade ein Dokument, das Text und mindestens ein Bild enthält (z. B. eine der vorbereiteten Dateien aus `./files`).
2. Initialisiere das RAG-System und indexiere das Dokument.
3. Stelle eine einfache Frage (Text-Query) und gib die Antwort aus.

**✅ Erledigt wenn:** Die Antwort enthält eine erkennbare Quellenangabe — Dateiname oder Bildpfad aus dem eigenen Dokument.

**Aufbau**

Indexiere mehrere Dokumente und beantworte Fragen, die Text- und Bildinhalte betreffen:

1. Lade mindestens 2 verschiedene Dokumente (Text + Bild) in das RAG-System.
2. Stelle mindestens 3 Fragen: eine reine Text-Query, eine Bild-Query und eine multimodale Query.
3. Gib bei jeder Antwort die gefundenen Quellen (Dokument, Seite oder Bildpfad) mit aus.

**✅ Erledigt wenn:** Alle drei Fragen (Text, Bild, kombiniert) liefern Antworten mit Quellenangaben — kein 'Ich weiß es nicht' bei erreichbaren Infos.

**Vertiefung**

Evaluiere die Qualität der multimodalen Retrieval-Ergebnisse:

1. Erstelle ein Eval-Set mit genau 5 Fragen (2 Text, 2 Bild, 1 multimodal) und den erwarteten Antworten.
2. Führe alle 5 Fragen über das RAG-System aus.
3. Bewerte jede Antwort mit einer Punktzahl (0 = falsch, 1 = teilweise, 2 = korrekt) und begründe die Bewertung.
4. Berechne den Durchschnittsscore und formuliere eine kurze Einschätzung der Systemqualität.

**✅ Erledigt wenn:** Das Eval-Set zeigt für alle fünf Fragen Score und Begründung; der Durchschnittsscore erscheint am Ende.

# B | Ähnlichkeitsmessung
---


<p><font color='black' size="5">
1. Text-Ähnlichkeit (semantisch)
</font></p>

  Embedding-Modell: OpenAI text-embedding-3-small

  Messmethode:
  - ChromaDB nutzt L2-Distanz (Euklidische Distanz) für Text-Embeddings
  - Werte: 0 = identisch, 2 = maximal entfernt
  - Konvertierung Distanz zu Ähnlichkeit: similarity = max(0, 1 - (score / 2))
  - Threshold: text_threshold: 1.2 (Zeile 51), Mindest-Ähnlichkeit: 0.3

  Verwendung in:
  - search_texts() - Text-Dokumente und Bildbeschreibungen durchsuchen
  - multimodal_search() - Kombinierte Suche

  ---

<p><font color='black' size="5">
2. Bild-Ähnlichkeit (visuell)
</font></p>

  Embedding-Modell: CLIP clip-ViT-B-32

  Messmethode:
  - ChromaDB nutzt Cosine-Distanz für Bild-Embeddings (Zeile 131: "hnsw:space": "cosine")
  - Werte: 0 = identisch, 2 = maximal entfernt
  - Konvertierung zu Ähnlichkeit:
  similarity = round(max(0, 1 - distance), 3)
  - Threshold: image_threshold: 0.8

  Verwendung in:
  - search_images() - Text → Bild Suche über CLIP   
  - search_similar_images() - Bild → Bild Suche  

  ---
  

<p><font color='black' size="5">
3. Cross-Modal-Retrieval (Text ↔ Bild)
</font></p>


  Methode: Indirekte Ähnlichkeit über Bildbeschreibungen

  Ablauf:
  1. Text → Bild: Text-Suche findet Bildbeschreibungen → verknüpfte Bilder werden abgerufen  
  2. Bild → Text: Bild-Suche findet ähnliche Bilder → deren Beschreibungen werden für semantische Textsuche verwendet

  Verknüpfung: Beide Collections sind über text_doc_id ↔ image_doc_id referenziert

  ---
  

<p><font color='black' size="5">
Zusammenfassung der Metriken:
</font></p>

  | Modalität | Embedding-Modell              | Distanzmetrik  | Threshold | Ähnlichkeitsbereich |
  |-----------|-------------------------------|----------------|-----------|---------------------|
  | Text      | OpenAI text-embedding-3-small | L2-Distanz     | 1.2       | 0.3 - 1.0           |
  | Bild      | CLIP ViT-B-32                 | Cosine-Distanz | 0.8       | 0.0 - 1.0           |

  Die Konvertierung 1 - (distance / 2) normalisiert beide Distanzmaße auf einen Ähnlichkeitswert von 0 bis 1, wobei 1 = maximale Ähnlichkeit bedeutet.

# C | Dokumente zum Weiterlesen
---




📚 Ergänzende Artikel aus der Kurs-Dokumentation:

- [RAG-Konzepte](https://ralf-42.github.io/GenAI/05-prompting-rag/rag-konzepte.html)
- [Embeddings](https://ralf-42.github.io/GenAI/03-grundlagen/embeddings.html)
- [Tokenizing & Chunking](https://ralf-42.github.io/GenAI/03-grundlagen/tokenizing-chunking.html)
- [ChromaDB](https://ralf-42.github.io/GenAI/06-frameworks/einsteiger-chromadb.html)
- [Multimodal: Bild](https://ralf-42.github.io/GenAI/09-multimodal/multimodal-bild.html)